# Workshop 1 — Web Scraping avec BeautifulSoup
**Story 10 – RegEx et Scraping**

Objectif (compétence **C1**) : automatiser l'extraction de données depuis une page web.
On scrape le site bac à sable **books.toscrape.com**, conçu pour s'entraîner légalement au scraping.

Déroulé demandé par le brief : **d'abord sans RegEx, puis avec RegEx** pour le nettoyage.

| Étape | Contenu |
|---|---|
| 1 | Télécharger une page (requête HTTP) |
| 2 | Extraire les données **sans RegEx** |
| 3 | Parcourir toutes les pages (pagination) |
| 4 | Nettoyer les données **avec RegEx** |
| 5 | Sauvegarder en CSV |
| 6 | Visualiser |

## Prérequis
Installer les bibliothèques (décommenter si besoin) :

In [ ]:
# !pip install requests beautifulsoup4 pandas matplotlib
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import matplotlib.pyplot as plt

## Étape 1 — Télécharger une page
On envoie une requête HTTP GET et on vérifie le code de statut (200 = OK).
On déclare un `User-Agent` par politesse et on récupère le HTML.

In [ ]:
BASE = 'https://books.toscrape.com/'
headers = {'User-Agent': 'Mozilla/5.0 (workshop pedagogique)'}

resp = requests.get(BASE, headers=headers, timeout=10)
print('Statut HTTP :', resp.status_code)

soup = BeautifulSoup(resp.text, 'html.parser')
print('Titre de la page :', soup.title.text.strip())

## Étape 2 — Extraire les données SANS RegEx
Chaque livre est dans une balise `<article class="product_pod">`.
On cible les éléments par leurs balises / classes, sans aucune expression régulière.

- titre : attribut `title` du lien dans le `<h3>`
- prix : texte de `<p class="price_color">` (ex. `£51.77`)
- note : 2e classe de `<p class="star-rating Three">`
- disponibilité : texte de `<p class="instock availability">`
- lien : attribut `href`

In [ ]:
def extraire_livres(soup):
    livres = []
    for art in soup.select('article.product_pod'):
        titre = art.h3.a['title']
        prix_txt = art.select_one('p.price_color').text
        note_txt = art.select_one('p.star-rating')['class'][-1]
        dispo_txt = art.select_one('p.instock.availability').text.strip()
        lien = art.h3.a['href']
        livres.append({
            'titre': titre,
            'prix_txt': prix_txt,
            'note_txt': note_txt,
            'dispo_txt': dispo_txt,
            'lien': lien,
        })
    return livres

livres_p1 = extraire_livres(soup)
print('Livres sur la page 1 :', len(livres_p1))
livres_p1[0]

## Étape 3 — Pagination : parcourir TOUTES les pages
Le bouton « next » est dans `<li class="next"><a href="...">`.
Tant qu'il existe, on suit le lien vers la page suivante.

In [ ]:
from urllib.parse import urljoin

url = BASE + 'catalogue/page-1.html'
tous = []
while url:
    r = requests.get(url, headers=headers, timeout=10)
    s = BeautifulSoup(r.text, 'html.parser')
    tous.extend(extraire_livres(s))
    suivant = s.select_one('li.next a')
    url = urljoin(url, suivant['href']) if suivant else None

print('Total de livres collectés :', len(tous))

## Étape 4 — Nettoyage AVEC RegEx
Les données brutes sont des chaînes de caractères. On utilise le module `re` pour en
extraire des valeurs exploitables (nombres, entiers).

- `prix` : on isole les chiffres et le point de `£51.77` → `51.77` (float)
- `note` : on convertit le mot (`Three`) en entier (`3`)
- `id_livre` : on extrait l'identifiant numérique présent dans l'URL

> Bonne pratique : on cible d'abord la bonne balise avec BeautifulSoup, **puis** on affine avec une RegEx.

In [ ]:
NOTES = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

def prix_to_float(txt):
    m = re.search(r'[\d.]+', txt)        # garde chiffres et point
    return float(m.group()) if m else None

def note_to_int(mot):
    return NOTES.get(mot)

def id_depuis_url(href):
    m = re.search(r'_(\d+)/', href)       # ..._1000/index.html -> 1000
    return int(m.group(1)) if m else None

# Démonstration rapide des RegEx
print(prix_to_float('£51.77'), note_to_int('Three'), id_depuis_url('catalogue/a-light_1000/index.html'))

On applique ces fonctions à l'ensemble des données et on construit un DataFrame propre.

In [ ]:
df = pd.DataFrame(tous)
df['prix'] = df['prix_txt'].apply(prix_to_float)
df['note'] = df['note_txt'].apply(note_to_int)
df['id_livre'] = df['lien'].apply(id_depuis_url)

df_propre = df[['id_livre', 'titre', 'prix', 'note', 'dispo_txt']]
print(df_propre.dtypes)
df_propre.head()

### Bonus RegEx — extraire le stock d'une page détail
Sur la page d'un livre, la disponibilité s'écrit `In stock (22 available)`.
Une RegEx isole le nombre. (Sur la page liste, seul « In stock » apparaît.)

In [ ]:
def stock_disponible(txt):
    m = re.search(r'\((\d+)\s+available\)', txt)
    return int(m.group(1)) if m else None

print(stock_disponible('In stock (22 available)'))   # 22

## Étape 5 — Sauvegarder en CSV
On regroupe tout dans un seul fichier structuré (objectif du brief).

In [ ]:
df_propre.to_csv('livres.csv', index=False, encoding='utf-8')
print('Fichier livres.csv enregistré :', len(df_propre), 'lignes')

## Étape 6 — Visualisation
Quelques graphiques simples pour valider la collecte.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].hist(df_propre['prix'].dropna(), bins=20, color='#6C7BD6', edgecolor='white')
ax[0].set_title('Distribution des prix (£)')
ax[0].set_xlabel('Prix'); ax[0].set_ylabel('Nombre de livres')

df_propre['note'].value_counts().sort_index().plot.bar(ax=ax[1], color='#6C7BD6')
ax[1].set_title('Nombre de livres par note')
ax[1].set_xlabel('Note (etoiles)'); ax[1].set_ylabel('Nombre de livres')

plt.tight_layout()
plt.show()

## Conclusion
On a automatisé toute la chaîne : **télécharger → parser → extraire (sans RegEx) → nettoyer (avec RegEx) → stocker → visualiser**.

**Transposition à l'exemple du brief (Fast Track 100)** : le principe est identique. On remplace
l'URL et les sélecteurs CSS (`select`, `select_one`) par ceux du tableau cible, puis on réutilise
les mêmes RegEx de nettoyage (prix, pourcentages, années).

**Éthique / cadre légal** : toujours consulter le `robots.txt` du site, respecter ses conditions
d'utilisation, limiter la fréquence des requêtes, et préférer une API officielle quand elle existe.